# CellChat

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # CellTalk
        library(CellChat), 
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        
        # Plotting 
        library(ComplexHeatmap), 
        library(circlize), 
        library(viridis), 
        library(ggplotify), 
        library(ggrepel), 
        library(cowplot), 
            
        # Pyhton compatibility
        library(reticulate)
        
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20220623HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
ht_opt$message=FALSE # ComplexHeatmap 

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [ ]:
# Source files 
source("plotting_global.R")

In [ ]:
# Plotting Theme
ggplot2::theme_set(theme_global_set()) # From project global source()

# Parameter and files 

In [ ]:
so_file <- "data/object/pp_subset.rds"

In [ ]:
grouping_var="cell_type_leiden_subset"

# Import 

In [70]:
so <- readRDS(so_file)

An object of class Seurat 
11874 features across 60019 samples within 1 assay 
Active assay: RNA (11874 features, 0 variable features)
 1 dimensional reduction calculated: umap

In [71]:
# Store data as h5ad 
adata <- import("anndata", as="adata", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)

adata <- adata$read_h5ad('data/object/raw.h5ad')

In [72]:
cnt <- py_to_r(adata$X$T) 
colnames(cnt) <- py_to_r(adata$obs_names$to_list())
rownames(cnt) <- py_to_r(adata$var_names$to_list())

In [73]:
cnt <- cnt[, colnames(so)]

In [74]:
so <- CreateSeuratObject(counts=cnt, meta.data=so@meta.data)

# Run CellChat pre-processing 

In [76]:
# Pre-process cellchat objects for each treatment condition 
cellchat_pp <- function(so, group_by, file) {
    
    # Normalized expression matrix 
    so <- NormalizeData(so)
    norm <- GetAssayData(so, assay="RNA", slot="data")
    
    # Get meta data 
    meta <- so@meta.data
    meta <- droplevels(meta)
    
    # Run CellChat pipeline
    cellchat <- createCellChat(object=norm, meta=meta, group.by=group_by)
    cellchat <- setIdent(cellchat, ident.use=group_by)
    cellchat@DB <- CellChatDB.human # Add DB slot with interaction, complex, cofactor, and geneInfo data frames 
    cellchat <- subsetData(cellchat) # Subset data for signaling genes 
    # cellchat <- identifyOverExpressedGenes(cellchat, thresh.p=1) # Identify DE signaling genes stored in object@var.features. 
    cellchat@var.features <- list(features=rownames(cellchat@data))
    cellchat <- identifyOverExpressedInteractions(cellchat) # Filter interaction based on over expressed genes. Adds @LR$LRsig which is a pruned persion of cellchat@DB$interaction
    cellchat <- projectData(cellchat, PPI.human) # Project protein data 
    cellchat <- computeCommunProb(cellchat, raw.use=FALSE, population.size=TRUE, type="truncatedMean", trim=0.01) # Using the information from DB compute interaction strength (probability) in object@net$prob and associated p-values in object@net$pval. 
    cellchat <- computeCommunProbPathway(cellchat) # Summarise L-R interaction on pathway level. Return object@netP$prob is the pathway probability and significant pathways are stored in object@netP$pathways
    cellchat <- aggregateNet(cellchat) # Updates object@net with counts and weights 
    cellchat <- netAnalysis_computeCentrality(cellchat)
    
    # Save result 
    saveRDS(cellchat, file)
    
}

# Run CellChat for Skin data with subset label per time point

In [77]:
# cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Baseline"), "cell_type_leiden_subset", "data/object/cellchat/skin_baseline.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Tx"), "cell_type_leiden_subset", "data/object/cellchat/skin_tx.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D14"), "cell_type_leiden_subset", "data/object/cellchat/skin_d14.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D100"), "cell_type_leiden_subset", "data/object/cellchat/skin_d100.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="GVHD"), "cell_type_leiden_subset", "data/object/cellchat/skin_gvhd.rds")

# Run CellChat for Skin data with subset label and PAGA annotation per time point

## Get leiden annotation from CD4+ and CD8+ subset DPT analysis

In [ ]:
ad <- import("anndata", as="adata", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)

In [ ]:
adata_0<- ad$read_h5ad("data/object/pseudotime/scvi_pt_0.h5ad")
leiden_0 <- data.frame(leiden=py_to_r(adata_0$obs$leiden$to_list()))
rownames(leiden_0) <- py_to_r(adata_0$obs_names$to_list())
leiden_0 <- leiden_0[leiden_0$leiden %in% c("0", "14", "3", "8"), , drop=FALSE]

In [ ]:
adata_1<- ad$read_h5ad("data/object/pseudotime/scvi_pt_1.h5ad")
leiden_1 <- data.frame(leiden=py_to_r(adata_1$obs$leiden$to_list()))
rownames(leiden_1) <- py_to_r(adata_1$obs_names$to_list())
leiden_1 <- leiden_1[leiden_1$leiden %in% c("1", "2", "4", "8"), , drop=FALSE]

In [ ]:
leiden <- rbind(leiden_0, leiden_1)
colnames(leiden) <- "leiden_paga"

## Add leiden annotation to Seurat object and correct cell type labels 

In [ ]:
so <- AddMetaData(so, leiden)

In [ ]:
so$cell_type_leiden_subset <- ifelse(!is.na(so$leiden_paga) & so$cell_type_leiden_subset!="Cycling", paste0(so$cell_type_leiden_subset, paste0("*'", so$leiden_paga, "'")), as.character(so$cell_type_leiden_subset))

## Run cell chat 

In [ ]:
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Baseline"), "cell_type_leiden_subset", "data/object/cellchat/skin_baseline_paga.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Tx"), "cell_type_leiden_subset", "data/object/cellchat/skin_tx_paga.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D14"), "cell_type_leiden_subset", "data/object/cellchat/skin_d14_paga.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D100"), "cell_type_leiden_subset", "data/object/cellchat/skin_d100_paga.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="GVHD"), "cell_type_leiden_subset", "data/object/cellchat/skin_gvhd_paga.rds")

# Run CellChat for Skin data with Th and Tc CCR6 IL7R phenotype label 

In [ ]:
cell_type_all <- unique(so$cell_type_leiden_subset)

In [ ]:
tcell_c_dp <- "T[\'c, memory/effector\']*\'4\'"
tcell_c_dn <- grep("c, memory/effector", cell_type_all, value=TRUE)[grep("c, memory/effector", cell_type_all, value=TRUE)!=tcell_c_dp]
    
tcell_h_dp <- "T[\'h, memory/effector\']*\'0\'"
tcell_h_dn <- grep("h, memory/effector", cell_type_all, value=TRUE)[grep("h, memory/effector", cell_type_all, value=TRUE)!=tcell_h_dp]
    
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_c_dp] <- "T['c, CCR6+ IL7R+']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_c_dn] <- "T['c, CCR6- IL7R-']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_h_dp] <- "T['h, CCR6+ IL7R+']"
so$cell_type_leiden_subset[so$cell_type_leiden_subset %in% tcell_h_dn] <- "T['h, CCR6- IL7R-']"

In [ ]:
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Baseline"), "cell_type_leiden_subset", "data/object/cellchat/skin_baseline_ccr6_il7r.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Tx"), "cell_type_leiden_subset", "data/object/cellchat/skin_tx_ccr6_il7r.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D14"), "cell_type_leiden_subset", "data/object/cellchat/skin_d14_ccr6_il7r.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D100"), "cell_type_leiden_subset", "data/object/cellchat/skin_d100_ccr6_il7r.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="GVHD"), "cell_type_leiden_subset", "data/object/cellchat/skin_gvhd_ccr6_il7r.rds")

# Session Info 

In [ ]:
sessionInfo()